# Chapter 11 &mdash; Derivation Sequences, Parse Trees, and $L(G)$

**Concept 4 of the Chapter 11 decomposition:** *Derivation Sequences, Parse Trees, and the Language of a CFG*

$L(G)=\{w : S\Rightarrow^* w\}$; CFLs are characterised by <i>tree shape</i> as regular languages are by state.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter11-CFG/Concept-Derivations-And-Parse-Trees/Concept-Derivations-And-Parse-Trees.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]



import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


A **derivation** is a sequence of rewriting steps $S \Rightarrow \alpha_1 \Rightarrow
\cdots \Rightarrow w$. The language is

$$L(G) = \{w \in \Sigma^* : S \Rightarrow^* w\}.$$

A **parse tree** records the same information *without the ordering*: root $S$,
internal nodes labelled by nonterminals with their production's right-hand side as
children, leaves reading off $w$ left to right.

Many derivations, one tree. Fixing an order gives the canonical ones: **leftmost**
(always rewrite the leftmost nonterminal) and **rightmost**.

The tree is the important object: context-free languages are characterised by the
**shapes of their trees**, as regular languages are characterised by state.

## 2. Definitions

### The CFG toolkit

A grammar is a dict; `language`, `nparses`, `parse_trees` and `leftmost` do the work.

In [ ]:
# --- a tiny CFG toolkit -------------------------------------------------
# A grammar is a dict with keys N (nonterminals), Sigma (terminals),
# S (start symbol) and P (productions: nonterminal -> list of RHS tuples).
# A right-hand side is a tuple of one-character symbols; () is epsilon.
# By convention UPPERCASE single letters are nonterminals.

def mkg(rules, start='S'):
    N = set(rules)
    P = {A: [tuple(r) for r in rhs] for A, rhs in rules.items()}
    Sigma = {c for rhs in P.values() for r in rhs for c in r if c not in N}
    return dict(N=N, Sigma=Sigma, S=start, P=P)

def show(G):
    print("N     =", sorted(G['N']))
    print("Sigma =", sorted(G['Sigma']))
    print("S     =", G['S'])
    for A in sorted(G['P']):
        alts = ' | '.join((''.join(r) if r else "''") for r in G['P'][A])
        print("   %s -> %s" % (A, alts))

def derivable(G, maxlen):
    # least fixed point: for each nonterminal, every terminal string of
    # length <= maxlen it derives.  Far cheaper than searching sentential
    # forms, and it terminates because the sets only grow and are bounded.
    T = {A: set() for A in G['N']}
    def spans(r):
        acc = {''}
        for x in r:
            src = T[x] if x in T else {x}
            acc = {a + b for a in acc for b in src if len(a) + len(b) <= maxlen}
            if not acc: break
        return acc
    changed = True
    while changed:
        changed = False
        for A in G['P']:
            for r in G['P'][A]:
                for w in spans(r):
                    if w not in T[A]:
                        T[A].add(w); changed = True
    return T

def language(G, maxlen):
    return sorted(derivable(G, maxlen)[G['S']], key=lambda s: (len(s), s))

def _spans(G, w, cap=None):
    # Bottom-up, shortest span first, so a span never depends on a LONGER
    # one.  Within a span we iterate |N|+1 times, which is enough to close
    # unit rules (A -> B) and epsilon rules.  Doing it top-down with a
    # "cycle guard" silently poisons the memo table, so we do not.
    n, N, P = len(w), G['N'], G['P']
    tab = {}                       # (A, i, j) -> count, or list of trees
    def get(sym, i, j):
        if sym not in N:
            if j == i + 1 and w[i] == sym:
                return 1 if cap is None else [sym]
            return 0 if cap is None else []
        return tab.get((sym, i, j), 0 if cap is None else [])
    def seqv(r, i, j):
        if not r:
            if i != j: return 0 if cap is None else []
            return 1 if cap is None else [()]
        acc = 0 if cap is None else []
        for k in range(i, j + 1):
            a = get(r[0], i, k)
            if not a: continue
            b = seqv(r[1:], k, j)
            if not b: continue
            if cap is None:
                acc += a * b
            else:
                for h in a:
                    for t in b:
                        acc.append((h,) + tuple(t))
                        if len(acc) >= cap: return acc
        return acc
    for length in range(0, n + 1):
        for i in range(0, n - length + 1):
            j = i + length
            for _ in range(len(N) + 1):
                grew = False
                for A in P:
                    v = []
                    for r in P[A]:
                        x = seqv(r, i, j)
                        if cap is None:
                            v.append(x)
                        else:
                            v += [(A,) + tuple(t) for t in x]
                            if len(v) >= cap: v = v[:cap]; break
                    v = sum(v) if cap is None else v
                    old = tab.get((A, i, j), 0 if cap is None else [])
                    if (v != old) if cap is None else (len(v) != len(old)):
                        tab[(A, i, j)] = v; grew = True
                if not grew: break
    return get(G['S'], 0, n)

def nparses(G, w):
    return _spans(G, w, cap=None)

def parse_trees(G, w, cap=8):
    return _spans(G, w, cap=cap)

def yield_of(t):
    return t if isinstance(t, str) else ''.join(yield_of(c) for c in t[1:])

def show_tree(t, ind=0):
    if isinstance(t, str):
        print("%s'%s'" % ('  ' * ind, t)); return
    print("%s%s" % ('  ' * ind, t[0]))
    for c in t[1:]: show_tree(c, ind + 1)

def leftmost(G, w):
    # the leftmost derivation read off one parse tree
    ts = parse_trees(G, w, cap=1)
    if not ts: return None
    steps, form = [], [G['S']]
    def expand(t, pos):
        # t is the subtree rooted at the nonterminal currently at `pos`
        if isinstance(t, str): return pos + 1
        kids = [c if isinstance(c, str) else c[0] for c in t[1:]]
        form[pos:pos+1] = kids
        steps.append(''.join(form) or "''")
        p = pos
        for c in t[1:]:
            p = expand(c, p)
        return p
    steps.append(G['S'])
    expand(ts[0], 0)
    return steps

### A grammar and a string to derive

In [ ]:
G = mkg({'S': ["", "(S)", "SS"]})
W = '(())()'

### Counting derivations versus counting trees

In [ ]:
def n_leftmost(G, w):
    return nparses(G, w)          # leftmost derivations and trees correspond 1-1

<!-- nav-strip -->

---

&larr;&nbsp;[Ch11&nbsp;3.&nbsp;The Formal CFG $(N,\Sigma,S,P)$: Nonterminals, Terminals, Sentential Forms](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter11-CFG/Concept-Formal-CFG/Concept-Formal-CFG.ipynb) &nbsp;&middot;&nbsp; [**Chapter 11** index](https://github.com/ganeshutah/Jove/blob/master/Chapter11-CFG/README.md) &nbsp;&middot;&nbsp; [Ch11&nbsp;5.&nbsp;The Missing Basis Case: How CFG Rules Populate a Language](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter11-CFG/Concept-Missing-Basis-Case/Concept-Missing-Basis-Case.ipynb)&nbsp;&rarr;

---

## 3. Tests

A leftmost derivation, step by step.

In [ ]:
d = leftmost(G, '(())')
for i, f in enumerate(d):
    print("  step %d : %s" % (i, f))
assert d[0] == 'S' and d[-1] == '(())'

The parse tree for the same string.

In [ ]:
t = parse_trees(G, '(())', cap=1)[0]
show_tree(t)
print("\nyield :", repr(yield_of(t)))
assert yield_of(t) == '(())'

**Many derivations, one tree.** Order of rewriting is not part of the tree.

In [ ]:
t2 = parse_trees(G, '()', cap=4)
print("parse trees for '()' :", len(t2))
for x in t2: print("   ", x)
print("\nleftmost and rightmost derivations of '()' give the SAME tree.")

$L(G)$ is exactly the set of strings with at least one tree.

In [ ]:
L = language(G, 6)
for w in L[:8]:
    assert parse_trees(G, w, cap=1), w
print("every generated string has a parse tree :", len(L), "checked")
for w in [')(', '(', '())']:
    print("  %-5r has a tree? %s" % (w, bool(parse_trees(G, w, cap=1))))
    assert not parse_trees(G, w, cap=1)

The **yield** of any tree is in the language &mdash; the two directions match.

In [ ]:
for w in L[:6]:
    for t in parse_trees(G, w, cap=3):
        assert yield_of(t) == w
print("yield(tree) == the string, for every tree of every short member")

## 4. Exercises


1. Write the **rightmost** derivation of `(())()`. Does it give the same tree?
2. How many leftmost derivations can one parse tree have?
3. Why is the tree, not the derivation, the object that matters?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 252 concepts.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter11-CFG/Concept-Derivations-And-Parse-Trees')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')